In [1]:
from benchmark_pipeline.path import DATA
from benchmark_pipeline.config import (
    FEATURES,
    CITY_BBOXES,
    CITY_ECRS,
)
from benchmark_pipeline.generator.features import *

from benchmark_pipeline.loader.osm_loaders import load_tables
from benchmark_pipeline.loader.data_cleaning.name_matching import name_matching_cleaning, make_df_all
from benchmark_pipeline.generator.composite.generator_qcomposite import make_question_features, make_question_spatial
from benchmark_pipeline.generator.composite.format_samples import format_composit_samples, split_spatial_questions
from benchmark_pipeline.generator.register import register, BenchmarkConfig
from benchmark_pipeline.generator.multihop.generator_multihop import make_multihop_dataset
from benchmark_pipeline.generator.relational.generator_relational import make_relational_dataset

In [2]:
import time

import numpy as np

OVERWRITE = False   # True regenerates the cities already done, False skips them

for city_name, bbox in CITY_BBOXES.items():
    city_start = time.perf_counter()
    print(f"\n=== [{list(CITY_BBOXES).index(city_name) + 1}/{len(CITY_BBOXES)}] {city_name} ===")

    if not OVERWRITE and (DATA / city_name / "question" / "test" / "multihop" / "df_multihop.parquet").exists():
        print("    déjà générée, ignorée (OVERWRITE = False)")
        continue
    conf = BenchmarkConfig(name=city_name, bbox=bbox, ecrs=CITY_ECRS[city_name], rows_target=[])

    print("1/6 chargement OSM...")
    step_start = time.perf_counter()
    df_osm, df_street, df_area = load_tables(conf.bbox, conf.ecrs, city_name=conf.name, use_cache=conf.use_cache)
    print(f"    {len(df_osm)} POIs, {len(df_street)} rues, {len(df_area)} zones ({time.perf_counter() - step_start:.0f} s)")

    print("2/6 dédoublonnage des noms...")
    if conf.desambiguous_name:
        df_all, (df_osm_clean, df_street_clean, df_area_clean) = name_matching_cleaning(df_osm, df_street, df_area)
    else: 
        print("Les noms sont gardés ambigus.")
        df_all, (df_osm_clean, df_street_clean, df_area_clean) = make_df_all(df_osm, df_street, df_area), (df_osm, df_street, df_area)
    print(f"    gardés : {len(df_osm_clean)} POIs, {len(df_street_clean)} rues, {len(df_area_clean)} zones")

    print("3/6 questions spatiales, coupées en train et test...")
    step_start = time.perf_counter()
    df_geo = make_question_spatial(df_osm_clean, df_area_clean, df_street_clean, nb_q_by_temp=conf.nb_q_by_temp,
                                   seed=conf.seed)
    geo_train, geo_test = split_spatial_questions(df_geo, conf.ratio_test, np.random.default_rng(conf.seed))
    geo_by_split = {"train": geo_train, "test": geo_test}
    print(f"    {len(df_geo)} questions spatiales : {len(geo_train)} train, {len(geo_test)} test "
          f"({time.perf_counter() - step_start:.0f} s)")

    print("4/6 attributs, dans chaque branche...")
    step_start = time.perf_counter()
    questions_by_split = {
        split: make_question_features(
            df_geo_split,
            df_osm_clean,
            FEATURES,
            conf.ratio_features,
            conf.nb_q_by_feat,
            conf.part_with_features,
            seed=conf.seed,
            is_paraphrase=conf.is_paraphrase,
        )
        for split, df_geo_split in geo_by_split.items()
    }
    print(f"    {len(questions_by_split['train'])} questions train, {len(questions_by_split['test'])} test "
          f"({time.perf_counter() - step_start:.0f} s)")

    print("5/6 enregistrement des questions simples...")
    step_start = time.perf_counter()
    register(
        DATA / conf.name,
        questions_by_split,
        df_osm_clean,
        df_street_clean,
        df_area_clean,
        df_all,
        conf,
        format_composit_samples,
        overwritte=True
    )
    print(f"    écrit dans {DATA / conf.name} ({time.perf_counter() - step_start:.0f} s)")

    print("6/6 questions multihop et relationnelles, dans chaque branche...")
    step_start = time.perf_counter()
    # starting questions of the multihop shared between the branches like the questions: 45 train, 5 test for 50
    nb_seeds_test = max(1, round(conf.nb_seed_questions * conf.ratio_test))
    nb_seeds = {"train": conf.nb_seed_questions - nb_seeds_test, "test": nb_seeds_test}
    for split, df_question in questions_by_split.items():
        make_multihop_dataset(
            nb_seeds[split],
            DATA / conf.name / "question" / split / "multihop",
            conf.multihop_template,
            df_question,
            df_osm_clean,
            df_street_clean,
            df_area_clean,
            seed=conf.seed,
            nb_q_by_feat=conf.nb_q_by_feat,
            ratio_features=conf.ratio_features,
            part_with_features=conf.part_with_features,
            max_questions_per_node=conf.max_questions_per_node,
            max_new_queries=conf.max_new_queries,
            lateral_area_buffer=conf.lateral_area_buffer,
            lateral_nb_q_by_temp=conf.lateral_nb_q_by_temp,
            lateral_nb_q_by_feat=conf.lateral_nb_q_by_feat,
        )
        """
    make_relational_dataset(
        conf.name,
        df_osm_clean,
        nb_questions=conf.relational_nb_questions,
        nb_pois=conf.relational_nb_pois,
        nb_relations=conf.relational_nb_relations,
        path_length=conf.relational_path_length,
        seed=conf.seed,
        ratio_test=conf.ratio_test,
        format_direction=conf.relational_format_direction,
    )"""
    print(f"    fait ({time.perf_counter() - step_start:.0f} s)")
    print(f"=== {city_name} terminé en {(time.perf_counter() - city_start) / 60:.1f} min ===")


=== [1/9] Paris ===
1/6 chargement OSM...
lecture cache raw_b252bd177ead.parquet


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:122: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


lecture cache raw_da2c65baf6d5.parquet
75 morceaux de rue recollés
lecture cache raw_d0aebc023b34.parquet
    18255 POIs, 8049 rues, 1217 zones (5 s)
2/6 dédoublonnage des noms...
Les noms sont gardés ambigus.
    gardés : 18255 POIs, 8049 rues, 1217 zones
3/6 questions spatiales, coupées en train et test...
    8157 questions spatiales : 7342 train, 815 test (27 s)
4/6 attributs, dans chaque branche...
    51108 questions train, 5705 test (17 s)
5/6 enregistrement des questions simples...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Paris (37 s)
6/6 questions multihop et relationnelles, dans chaque branche...
    fait (725 s)
=== Paris terminé en 13.5 min ===

=== [2/9] New York ===
1/6 chargement OSM...
lecture cache raw_e634ce027a84.parquet


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:122: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


lecture cache raw_281a5a03e39d.parquet
2475 morceaux de rue recollés
lecture cache raw_1a194243d7af.parquet
    15932 POIs, 26034 rues, 2565 zones (14 s)
2/6 dédoublonnage des noms...
Les noms sont gardés ambigus.
    gardés : 15932 POIs, 26034 rues, 2565 zones
3/6 questions spatiales, coupées en train et test...
    7208 questions spatiales : 6488 train, 720 test (41 s)
4/6 attributs, dans chaque branche...
    34540 questions train, 3772 test (15 s)
5/6 enregistrement des questions simples...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/New York (23 s)
6/6 questions multihop et relationnelles, dans chaque branche...
    fait (463 s)
=== New York terminé en 9.3 min ===

=== [3/9] Tokyo ===
1/6 chargement OSM...
lecture cache raw_78551def9a43.parquet


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:122: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


... (11 outputs left out)

/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:122: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


10 morceaux de rue recollés
lecture cache raw_2aca91e6e484.parquet
    930 POIs, 664 rues, 28 zones (1 s)
2/6 dédoublonnage des noms...
Les noms sont gardés ambigus.
    gardés : 930 POIs, 664 rues, 28 zones
3/6 questions spatiales, coupées en train et test...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/generator/geospatial/point_between.py:39: RuntimeWarning: invalid value encountered in divide
  along = (px * abx + py * aby) / ab_len
/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/generator/geospatial/point_between.py:40: RuntimeWarning: invalid value encountered in divide
  cross_m = np.abs(px * aby - py * abx) / ab_len


    3912 questions spatiales : 3521 train, 391 test (4 s)
4/6 attributs, dans chaque branche...
    18014 questions train, 2010 test (8 s)
5/6 enregistrement des questions simples...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Marrakesh (9 s)
6/6 questions multihop et relationnelles, dans chaque branche...
    fait (140 s)
=== Marrakesh terminé en 2.7 min ===
